In [98]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

In [5]:
data = pd.read_csv("../data/Bengaluru_House_Data.csv")
data

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00
...,...,...,...,...,...,...,...,...,...
13315,Built-up Area,Ready To Move,Whitefield,5 Bedroom,ArsiaEx,3453,4.0,0.0,231.00
13316,Super built-up Area,Ready To Move,Richards Town,4 BHK,NaN,3600,5.0,NaN,400.00
13317,Built-up Area,Ready To Move,Raja Rajeshwari Nagar,2 BHK,Mahla T,1141,2.0,1.0,60.00
13318,Super built-up Area,18-Jun,Padmanabhanagar,4 BHK,SollyCl,4689,4.0,1.0,488.00


In [6]:
df = data.copy()

In [7]:
df.isnull().sum()

area_type          0
availability       0
location           1
size              16
society         5502
total_sqft         0
bath              73
balcony          609
price              0
dtype: int64

In [8]:
df.columns

Index(['area_type', 'availability', 'location', 'size', 'society',
       'total_sqft', 'bath', 'balcony', 'price'],
      dtype='str')

In [9]:
X = df.drop('price', axis=1)
y = df['price']

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [11]:
X_train = X_train.drop('society', axis=1)
X_test = X_test.drop('society', axis=1)

In [12]:

X_train.isnull().sum()


area_type         0
availability      0
location          1
size             14
total_sqft        0
bath             65
balcony         499
dtype: int64

In [13]:
imputer = SimpleImputer()

In [14]:
numerical_cols = ['total_sqft', 'bath', 'balcony']
categorical_cols = ['area_type', 'availability', 'location', 'size']

In [15]:
num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')

In [16]:
def convert_string(x):
    try:
        x=str(x)

        if '-' in x:
            a, b = x.split('-')
            return ((float(a)+float(b))/2)
        
        return float(x)
    
    except:
        return None

In [17]:
X_train['total_sqft'] = X_train['total_sqft'].apply(convert_string)
X_test['total_sqft'] = X_test['total_sqft'].apply(convert_string)

In [18]:
X_train['total_sqft'].isnull().sum()

np.int64(34)

In [19]:
X_train = X_train.dropna(subset=['total_sqft'])
X_test = X_test.dropna(subset=['total_sqft'])

In [20]:
X_train[numerical_cols] = num_imputer.fit_transform(X_train[numerical_cols])
X_test[numerical_cols] = num_imputer.transform(X_test[numerical_cols])

In [21]:
X_train.isnull().sum()

area_type        0
availability     0
location         1
size            14
total_sqft       0
bath             0
balcony          0
dtype: int64

In [22]:
X_train[categorical_cols] = cat_imputer.fit_transform(X_train[categorical_cols])
X_test[categorical_cols] = cat_imputer.transform(X_test[categorical_cols])

In [23]:
X_train['availability'] = X_train['availability'].str.strip().str.lower().apply(lambda x : 1 if x=='ready to move' else 0)
X_test['availability'] = X_test['availability'].str.strip().str.lower().apply(lambda x : 1 if x=='ready to move' else 0)

In [24]:
X_train

,area_type,availability,location,size,total_sqft,bath,balcony
3411,Super built-up Area,1,Lingadheeranahalli,3 BHK,1530.0,2.0,2.0
9142,Super built-up Area,1,Cooke Town,2 BHK,1310.0,2.0,2.0
1971,Super built-up Area,1,Raja Rajeshwari Nagar,3 BHK,1530.0,3.0,2.0
2608,Plot Area,1,Banashankari,4 Bedroom,2400.0,3.0,2.0
9635,Built-up Area,0,Kanakapura,2 BHK,1017.0,2.0,1.0
...,...,...,...,...,...,...,...
11964,Plot Area,1,Varanasi,5 Bedroom,1200.0,4.0,1.0
5191,Super built-up Area,0,Kogilu,3 BHK,1559.0,3.0,2.0
5390,Super built-up Area,1,Garudachar Palya,2 BHK,1060.0,2.0,1.0
860,Plot Area,1,Raja Rajeshwari Nagar,6 Bedroom,1200.0,4.0,2.0


In [25]:
X_train['size_num'] = X_train['size'].str.extract(r'(\d+)').astype(int)
X_test['size_num'] = X_test['size'].str.extract(r'(\d+)').astype(int)

In [26]:
X_train

,area_type,availability,location,size,total_sqft,bath,balcony,size_num
3411,Super built-up Area,1,Lingadheeranahalli,3 BHK,1530.0,2.0,2.0,3
9142,Super built-up Area,1,Cooke Town,2 BHK,1310.0,2.0,2.0,2
1971,Super built-up Area,1,Raja Rajeshwari Nagar,3 BHK,1530.0,3.0,2.0,3
2608,Plot Area,1,Banashankari,4 Bedroom,2400.0,3.0,2.0,4
9635,Built-up Area,0,Kanakapura,2 BHK,1017.0,2.0,1.0,2
...,...,...,...,...,...,...,...,...
11964,Plot Area,1,Varanasi,5 Bedroom,1200.0,4.0,1.0,5
5191,Super built-up Area,0,Kogilu,3 BHK,1559.0,3.0,2.0,3
5390,Super built-up Area,1,Garudachar Palya,2 BHK,1060.0,2.0,1.0,2
860,Plot Area,1,Raja Rajeshwari Nagar,6 Bedroom,1200.0,4.0,2.0,6


In [27]:
encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')

In [28]:
X_train_encoded = encoder.fit_transform(X_train[['area_type']])
X_test_encoded = encoder.transform(X_test[['area_type']])

In [29]:
cols = encoder.get_feature_names_out(['area_type'])

In [30]:
train_encoded_df = pd.DataFrame(X_train_encoded, columns=cols, index=X_train.index)
test_encoded_df = pd.DataFrame(X_test_encoded, columns=cols, index=X_test.index)

In [31]:
X_train = pd.concat([X_train, train_encoded_df], axis='columns')
X_test = pd.concat([X_test, test_encoded_df], axis='columns')

In [32]:
X_train

,area_type,availability,location,size,total_sqft,bath,balcony,size_num,area_type_Carpet Area,area_type_Plot Area,area_type_Super built-up Area
3411,Super built-up Area,1,Lingadheeranahalli,3 BHK,1530.0,2.0,2.0,3,0.0,0.0,1.0
9142,Super built-up Area,1,Cooke Town,2 BHK,1310.0,2.0,2.0,2,0.0,0.0,1.0
1971,Super built-up Area,1,Raja Rajeshwari Nagar,3 BHK,1530.0,3.0,2.0,3,0.0,0.0,1.0
2608,Plot Area,1,Banashankari,4 Bedroom,2400.0,3.0,2.0,4,0.0,1.0,0.0
9635,Built-up Area,0,Kanakapura,2 BHK,1017.0,2.0,1.0,2,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
11964,Plot Area,1,Varanasi,5 Bedroom,1200.0,4.0,1.0,5,0.0,1.0,0.0
5191,Super built-up Area,0,Kogilu,3 BHK,1559.0,3.0,2.0,3,0.0,0.0,1.0
5390,Super built-up Area,1,Garudachar Palya,2 BHK,1060.0,2.0,1.0,2,0.0,0.0,1.0
860,Plot Area,1,Raja Rajeshwari Nagar,6 Bedroom,1200.0,4.0,2.0,6,0.0,1.0,0.0


In [33]:
X_test

,area_type,availability,location,size,total_sqft,bath,balcony,size_num,area_type_Carpet Area,area_type_Plot Area,area_type_Super built-up Area
8077,Built-up Area,1,Banjara Layout,2 Bedroom,1050.0,2.0,1.0,2,0.0,0.0,0.0
1602,Super built-up Area,1,Rajiv Nagar,3 BHK,1690.0,3.0,1.0,3,0.0,0.0,1.0
10498,Built-up Area,1,Hebbal,2 BHK,1100.0,2.0,1.0,2,0.0,0.0,0.0
3297,Plot Area,1,Munnekollal,6 Bedroom,1200.0,4.0,2.0,6,0.0,1.0,0.0
8893,Built-up Area,0,Choodasandra,4 Bedroom,2429.0,3.0,1.0,4,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
1082,Super built-up Area,0,Ananth Nagar,1 BHK,500.0,1.0,1.0,1,0.0,0.0,1.0
1671,Built-up Area,1,Herohalli,5 Bedroom,1800.0,6.0,0.0,5,0.0,0.0,0.0
4325,Super built-up Area,1,AECS Layout,3 BHK,1800.0,3.0,3.0,3,0.0,0.0,1.0
7375,Super built-up Area,1,Vinayaka Nagar,2 BHK,1200.0,2.0,2.0,2,0.0,0.0,1.0


In [34]:
X_train.columns

Index(['area_type', 'availability', 'location', 'size', 'total_sqft', 'bath',
       'balcony', 'size_num', 'area_type_Carpet  Area', 'area_type_Plot  Area',
       'area_type_Super built-up  Area'],
      dtype='str')

In [35]:
X_train = X_train.drop(['area_type', 'size'], axis=1)
X_test = X_test.drop(['area_type', 'size'], axis=1)

In [36]:
X_train

,availability,location,total_sqft,bath,balcony,size_num,area_type_Carpet Area,area_type_Plot Area,area_type_Super built-up Area
3411,1,Lingadheeranahalli,1530.0,2.0,2.0,3,0.0,0.0,1.0
9142,1,Cooke Town,1310.0,2.0,2.0,2,0.0,0.0,1.0
1971,1,Raja Rajeshwari Nagar,1530.0,3.0,2.0,3,0.0,0.0,1.0
2608,1,Banashankari,2400.0,3.0,2.0,4,0.0,1.0,0.0
9635,0,Kanakapura,1017.0,2.0,1.0,2,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...
11964,1,Varanasi,1200.0,4.0,1.0,5,0.0,1.0,0.0
5191,0,Kogilu,1559.0,3.0,2.0,3,0.0,0.0,1.0
5390,1,Garudachar Palya,1060.0,2.0,1.0,2,0.0,0.0,1.0
860,1,Raja Rajeshwari Nagar,1200.0,4.0,2.0,6,0.0,1.0,0.0


In [37]:
sqft_per_room = X_train['total_sqft'] / X_train['size_num']

In [38]:
outliers = (sqft_per_room >= 300) & (sqft_per_room <= 1500)

In [39]:
y_train = y_train.loc[X_train.index]

In [40]:
X_train = X_train[outliers]
y_train = y_train[outliers]

In [41]:
sqft_per_room_test = X_test['total_sqft'] / X_test['size_num']

outliers_test = (sqft_per_room_test >= 300) & (sqft_per_room_test <= 1500)

X_test = X_test[outliers_test]
y_test = y_test.loc[X_test.index]

In [42]:
X_train['bath_per_bhk'] = X_train['bath'] / X_train['size_num']
X_test['bath_per_bhk'] = X_test['bath'] / X_test['size_num']

In [43]:
X_train['sqft_per_bhk'] = X_train['total_sqft'] / X_train['size_num']
X_test['sqft_per_bhk'] = X_test['total_sqft'] / X_test['size_num']

In [44]:
X_train['total_sqft'] = np.log(X_train['total_sqft'])
X_test['total_sqft'] = np.log(X_test['total_sqft'])

In [45]:
X_train['is_luxury'] = ((X_train['size_num'] >= 4) & (X_train['total_sqft'] >= np.log(2000))).astype(int)
X_test['is_luxury'] = ((X_test['size_num'] >= 4) & (X_test['total_sqft'] >= np.log(2000))).astype(int)

In [46]:
X_train['total_rooms'] = X_train['size_num'] + X_train['bath'] + X_train['balcony']
X_test['total_rooms'] = X_test['size_num'] + X_test['bath'] + X_test['balcony']

In [47]:
price_per_sqft = y_train / np.exp(X_train['total_sqft']) 

loc_ppsf = price_per_sqft.groupby(X_train['location']).mean()
X_train['loc_ppsf'] = X_train['location'].map(loc_ppsf)
X_test['loc_ppsf'] = X_test['location'].map(loc_ppsf).fillna(loc_ppsf.mean())

In [54]:
from category_encoders import TargetEncoder

encoder = TargetEncoder(cols=['location'], smoothing=10)

X_train = encoder.fit_transform(X_train, y_train)

X_test = encoder.transform(X_test)

In [55]:
y_train_log = np.log(y_train)

In [56]:
X_train

,availability,location,total_sqft,bath,balcony,size_num,area_type_Carpet Area,area_type_Plot Area,area_type_Super built-up Area,bath_per_bhk,sqft_per_bhk,is_luxury,total_rooms,loc_ppsf
3411,1,110.384405,7.333023,2.0,2.0,3,0.0,0.0,1.0,0.666667,510.000000,0,7.0,0.065895
9142,1,154.060623,7.177782,2.0,2.0,2,0.0,0.0,1.0,1.000000,655.000000,0,6.0,0.110333
1971,1,66.516389,7.333023,3.0,2.0,3,0.0,0.0,1.0,1.000000,510.000000,0,8.0,0.046405
2608,1,100.763164,7.783224,3.0,2.0,4,0.0,1.0,0.0,0.750000,600.000000,1,9.0,0.062248
9635,0,66.435879,6.924612,2.0,1.0,2,0.0,0.0,0.0,1.000000,508.500000,0,5.0,0.045147
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5734,1,66.516389,7.118826,2.0,1.0,2,0.0,0.0,1.0,1.000000,617.500000,0,5.0,0.046405
11284,1,94.038738,7.539027,3.0,2.0,3,0.0,0.0,1.0,1.000000,626.666667,0,8.0,0.058850
5191,0,88.904359,7.351800,3.0,2.0,3,0.0,0.0,1.0,1.000000,519.666667,0,8.0,0.052173
5390,1,105.348816,6.966024,2.0,1.0,2,0.0,0.0,1.0,1.000000,530.000000,0,5.0,0.055389


In [57]:
scaler = StandardScaler()

In [58]:
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [59]:
X_train.index

Index([ 3411,  9142,  1971,  2608,  9635,  9611, 11875,   694, 11318,  6113,
       ...
       11636,  5578,  4426,   466,  6265,  5734, 11284,  5191,  5390,  7270],
      dtype='int64', length=9953)

In [60]:
y_train.index

Index([ 3411,  9142,  1971,  2608,  9635,  9611, 11875,   694, 11318,  6113,
       ...
       11636,  5578,  4426,   466,  6265,  5734, 11284,  5191,  5390,  7270],
      dtype='int64', length=9953)

In [61]:
model = LinearRegression()

In [62]:
scores = cross_val_score(model, X_train_scaled, y_train_log, cv=5)

In [63]:
scores.mean()

np.float64(0.821062287700548)

In [64]:
model.fit(X_train_scaled, y_train_log)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [65]:
train_score = model.score(X_train_scaled, y_train_log)
print(train_score)

0.823289097918598


In [66]:
y_pred = np.exp(model.predict(X_test_scaled))

In [67]:
rmse_score = root_mean_squared_error(y_test, y_pred)
r2_score_ = r2_score(y_test, y_pred)

In [68]:
r2_score_

0.6934347994957915

In [69]:
rmse_score

69.50665161229966

In [72]:
model2 = GradientBoostingRegressor(
    n_estimators=600,
    max_depth=3, 
    learning_rate=0.03,
    random_state=42, 
    min_samples_leaf=10,
    max_features=0.8,
    subsample=0.7
)

In [73]:
cv_score = cross_val_score(model2, X_train, y_train_log, cv=5)

In [74]:
cv_score

array([0.86742528, 0.86391458, 0.847639  , 0.864296  , 0.85791499])

In [75]:
cv_score.mean()

np.float64(0.8602379715865404)

In [76]:
model2.fit(X_train, y_train_log)

,"loss loss: {'squared_error', 'absolute_error', 'huber', 'quantile'}, default='squared_error'Loss function to be optimized. 'squared_error' refers to the squarederror for regression. 'absolute_error' refers to the absolute error ofregression and is a robust loss function. 'huber' is acombination of the two. 'quantile' allows quantile regression (use`alpha` to specify the quantile).See:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_quantile.py`for an example that demonstrates quantile regression for creatingprediction intervals with `loss='quantile'`.",'squared_error'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.",0.03
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",600
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",0.7
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'The function to measure the quality of a split. Supported criteria are""friedman_mse"" for the mean squared error with improvement score byFriedman, ""squared_error"" for mean squared error. The default value of""friedman_mse"" is generally the best as it can provide a betterapproximation in some cases... versionadded:: 0.18",'friedman_mse'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",10
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",3
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in thelef

In [77]:
train2_score = model2.score(X_train, y_train_log)

In [78]:
train2_score

0.8789595244551398

In [79]:
y_pred = np.exp(model2.predict(X_test))

In [80]:
rmse2 = root_mean_squared_error(y_test, y_pred)
r2score = r2_score(y_test, y_pred)

In [81]:
rmse2

59.064765220783876

In [82]:
r2score

0.7786257597040926

In [ ]:
model3 = XGBRegressor(
    n_estimators=1000,
    max_depth=4,          
    learning_rate=0.05,   
    subsample=0.8,
    colsample_bytree=0.8, 
    reg_alpha=0.1,        
    reg_lambda=2.0,       
    random_state=42
)

In [84]:
cv_score2 = cross_val_score(model3, X_train, y_train_log, cv=5)

In [85]:
cv_score2

array([0.87647542, 0.86965506, 0.85749319, 0.87270252, 0.8674158 ])

In [86]:
cv_score2.mean()

np.float64(0.8687483985260602)

In [87]:
model3.fit(X_train, y_train_log)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_meth

In [88]:
score3 = model3.score(X_train, y_train_log)
score3

0.9202659493767106

In [89]:
model3.fit(X_train, y_train_log)
y_pred = np.exp(model3.predict(X_test))

In [90]:
print("R2:", r2_score(y_test, y_pred))
print("RMSE:", root_mean_squared_error(y_test, y_pred))

R2: 0.7647892269735157
RMSE: 60.88264962020163


In [91]:
print("500L+ in train:", (y_train > 500).sum())
print("500L+ in test:", (y_test > 500).sum())
print("Total train size:", len(y_train))

# Cap at 500 lakhs
mask = y_train <= 500
X_train_capped = X_train[mask]
y_train_capped = y_train[mask]
y_train_log_capped = np.log(y_train_capped)

500L+ in train: 151
500L+ in test: 41
Total train size: 9953


In [92]:
model3.fit(X_train_capped, y_train_log_capped)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_meth

In [99]:
test_mask = y_test <= 500
y_pred_capped = np.exp(model3.predict(X_test[test_mask]))
print("R2 without outlier:", r2_score(y_test[test_mask], y_pred_capped))
print("RMSE without outlier:", root_mean_squared_error(y_test[test_mask], y_pred_capped))

R2 without outlier: 0.7940115361654878
RMSE without outlier: 35.90540475171206


In [94]:
cv_score3 = cross_val_score(model3, X_train_capped, y_train_log_capped, cv=5)

In [95]:
cv_score3

array([0.86127688, 0.86258754, 0.85712403, 0.86052487, 0.86295601])

In [96]:
cv_score3.mean()

np.float64(0.8608938656943668)

In [97]:
train_score3 = model3.score(X_train_capped, y_train_log_capped)
train_score3

0.9133942666675734

In [100]:
model4 = RandomForestRegressor(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=10,
    max_features=0.8,
    random_state=42,
)

In [101]:
cv_score4 = cross_val_score(model4, X_train_capped, y_train_log_capped, cv=5)

In [102]:
cv_score4

array([0.85043173, 0.85129156, 0.84683329, 0.84775871, 0.84949496])

In [103]:
cv_score4.mean()

np.float64(0.849162051134994)

In [104]:
model4.fit(X_train_capped, y_train_log_capped)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",10
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",0.8
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples

In [105]:
train4_score = model4.score(X_train_capped, y_train_log_capped)
train4_score

0.883549468256967

In [106]:
y_pred4 = np.exp(model4.predict(X_test[test_mask]))

In [107]:
print("R2:", r2_score(y_test[test_mask], y_pred4))
print("RMSE:", root_mean_squared_error(y_test[test_mask], y_pred4))

R2: 0.7912211335619252
RMSE: 36.14778120966301
